# Knowledge Graph Construction Pipeline
## Strwythura-Adapted Academic ERKG

**Logging:** All steps log to `logs/kg_pipeline_<timestamp>.log` (DEBUG) and console (INFO).

| Cell | Strwythura Part | Purpose |
|---|---|---|
| 1 | Setup | Ontology + GLiNER + spaCy + Logging |
| 2 | Part 1 (ER) | Backbone from structured CSV |
| 3 | Part 3 (Parse) | spaCy + GLiNER + lemma-key entity store |
| 4 | Part 1+4 | Entity Resolution (3-layer) |
| 5 | Part 4 (HITL) | LLM Curation (validate + enrich) |
| 6 | Part 5 (Distil) | Distillation + Entity Embeddings |
| 7 | Part 6 (DB) | Neo4j + Weaviate ingestion |
| 8 | Verification | 10 Cypher test queries |

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1: Setup & Academic Ontology
# ══════════════════════════════════════════════════════════════════════
import pandas as pd
import json, os, requests, time, warnings, hashlib, re, logging, sys
from collections import defaultdict, OrderedDict
from datetime import datetime
warnings.filterwarnings('ignore')

from neo4j import GraphDatabase
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.data import DataObject
from dotenv import load_dotenv
from gliner import GLiNER
import spacy

# ── Production Logging Setup ──
LOG_DIR = 'logs'
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, f'kg_pipeline_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')

logger = logging.getLogger('kg_pipeline')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

# File handler: DEBUG (captures everything for post-mortem debugging)
fh = logging.FileHandler(LOG_FILE, encoding='utf-8')
fh.setLevel(logging.DEBUG)
fh.setFormatter(logging.Formatter(
    '%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
))
logger.addHandler(fh)

# Console handler: DEBUG (shows all progress directly in notebook output)
ch = logging.StreamHandler(sys.stdout)
ch.setLevel(logging.DEBUG)
ch.setFormatter(logging.Formatter('%(levelname)-8s | %(message)s'))
logger.addHandler(ch)

# Suppress verbose HTTP connection logs from requests library
logging.getLogger('urllib3').setLevel(logging.WARNING)

# Pipeline timer utility
_cell_timers = {}
def start_cell(cell_name):
    _cell_timers[cell_name] = time.time()
    logger.info(f'{"="*60}')
    logger.info(f'START: {cell_name}')
    logger.info(f'{"="*60}')

def end_cell(cell_name, stats=None):
    elapsed = time.time() - _cell_timers.get(cell_name, time.time())
    logger.info(f'{"─"*60}')
    if stats:
        for k, v in stats.items():
            logger.info(f'  {k}: {v}')
    logger.info(f'✅ {cell_name} completed in {elapsed:.1f}s')
    logger.info(f'{"="*60}\n')

start_cell('Cell 1: Setup & Ontology')

load_dotenv('../../.env')
NEO4J_URI   = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER  = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASS  = os.getenv('NEO4J_PASS', 'rizkyyk123')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
WEAVIATE_HOST = os.getenv('WEAVIATE_HOST', 'localhost')
WEAVIATE_PORT = int(os.getenv('WEAVIATE_PORT', '8081'))
LLM_MODEL = 'nvidia/nemotron-3-super-120b-a12b:free'

logger.info(f'Config: NEO4J={NEO4J_URI}, WEAVIATE={WEAVIATE_HOST}:{WEAVIATE_PORT}')
logger.info(f'LLM Model: {LLM_MODEL}')
logger.debug(f'OpenRouter API key loaded: {"YES" if OPENROUTER_API_KEY else "MISSING!"}')

# ── Academic Ontology (equivalent to Strwythura's domain.ttl) ──
ONTOLOGY = {
    'node_types': {
        'Dosen': {'description': 'Internal faculty member'},
        'ExternalAuthor': {'description': 'Non-faculty co-author'},
        'Paper': {'description': 'Research publication'},
        'ProgramStudi': {'description': 'Study program / department'},
        'Fakultas': {'description': 'Faculty / college'},
        'Journal': {'description': 'Publication venue'},
        'Year': {'description': 'Publication year'},
        'Keyword': {'description': 'Author-assigned keyword'},
        'Method': {'description': 'Research method or algorithm'},
        'Model': {'description': 'ML/AI model architecture'},
        'Metric': {'description': 'Evaluation metric'},
        'Dataset': {'description': 'Dataset used in study'},
        'Problem': {'description': 'Research problem addressed'},
        'Task': {'description': 'Computational/research task'},
        'Field': {'description': 'Research domain/field'},
        'Tool': {'description': 'Software tool or framework'},
        'Innovation': {'description': 'Novel contribution'},
    },
    'edge_types': {
        'WRITES': ('Dosen/ExternalAuthor', 'Paper'),
        'MEMBER_OF': ('Dosen', 'ProgramStudi'),
        'PART_OF': ('ProgramStudi', 'Fakultas'),
        'PUBLISHED_YEAR': ('Paper', 'Year'),
        'PUBLISHED_IN': ('Paper', 'Journal'),
        'HAS_KEYWORD': ('Paper', 'Keyword'),
        'HAS_METHOD': ('Paper', 'Method'),
        'HAS_MODEL': ('Paper', 'Model'),
        'HAS_METRIC': ('Paper', 'Metric'),
        'HAS_DATASET': ('Paper', 'Dataset'),
        'ADDRESSES': ('Paper', 'Problem'),
        'HAS_TASK': ('Paper', 'Task'),
        'IN_FIELD': ('Paper', 'Field'),
        'HAS_TOOL': ('Paper', 'Tool'),
        'PROPOSES': ('Paper', 'Innovation'),
        'USES': ('Entity', 'Entity'),
    },
    'ner_labels': ['method', 'model', 'metric', 'dataset', 'problem', 'task', 'field', 'tool', 'innovation'],
    'ner_label_map': {
        'method': 'Method', 'model': 'Model', 'metric': 'Metric',
        'dataset': 'Dataset', 'problem': 'Problem', 'task': 'Task',
        'field': 'Field', 'tool': 'Tool', 'innovation': 'Innovation',
        'algorithm': 'Method', 'technique': 'Method', 'framework': 'Tool',
        'software': 'Tool', 'platform': 'Tool', 'evaluation metric': 'Metric',
        'research method': 'Method', 'scientific concept': 'Field',
        'technology': 'Tool', 'programming language': 'Tool',
    },
}

PRODI_FAKULTAS = {
    'S1 Teknik Informatika': 'Fakultas Teknik',
    'S1 Sistem Informasi': 'Fakultas Teknik',
    'S1 Pendidikan Teknologi Informasi': 'Fakultas Teknik',
    'S1 Teknik Elektro': 'Fakultas Teknik',
    'S2 Informatika': 'Fakultas Teknik',
    'S2 Pendidikan Teknologi Informasi': 'Pascasarjana',
    'S1 Kecerdasan Artifisial': 'FMIPA',
    'S1 Sains Data': 'FMIPA',
    'S1 Bisnis Digital': 'FEB',
    'D4 Manajemen Informatika': 'Vokasi',
}

logger.info(f'Ontology loaded: {len(ONTOLOGY["node_types"])} node types, {len(ONTOLOGY["edge_types"])} edge types')

# ── Helper Functions ──
def md5(text):
    return hashlib.md5(text.encode()).hexdigest()[:12]

def normalize_text(text):
    text = re.sub(r'[^\w\s]', '', str(text).lower().strip())
    return re.sub(r'\s+', ' ', text).strip()

def call_openrouter(prompt, max_retries=3):
    url = 'https://openrouter.ai/api/v1/chat/completions'
    headers = {'Authorization': f'Bearer {OPENROUTER_API_KEY}', 'Content-Type': 'application/json'}
    payload = {
        'model': LLM_MODEL,
        'messages': [{'role': 'user', 'content': prompt}],
        'temperature': 0.0, 'max_tokens': 3000,
        'response_format': {'type': 'json_object'}
    }
    for attempt in range(max_retries):
        try:
            res = requests.post(url, headers=headers, json=payload, timeout=60)
            if res.status_code != 200:
                logger.warning(f'OpenRouter HTTP {res.status_code}: {res.text[:200]}')
                if attempt < max_retries - 1: time.sleep(2 ** attempt)
                continue
            content = res.json()['choices'][0]['message']['content'].strip()
            if content.startswith('```'): content = content.split('\n', 1)[1].rsplit('```', 1)[0]
            logger.debug(f'LLM response length: {len(content)} chars')
            return json.loads(content)
        except json.JSONDecodeError as e:
            logger.warning(f'LLM JSON parse error (attempt {attempt+1}): {e}')
            logger.debug(f'Raw content: {content[:300]}')
            if attempt < max_retries - 1: time.sleep(2 ** attempt)
            else: return {}
        except Exception as e:
            logger.warning(f'LLM API error (attempt {attempt+1}): {type(e).__name__}: {e}')
            if attempt < max_retries - 1: time.sleep(2 ** attempt)
            else: return {}
    return {}

# ── Load NLP models ──
logger.info('Loading spaCy en_core_web_sm...')
nlp = spacy.load('en_core_web_sm')
logger.info('Loading GLiNER urchade/gliner_multi-v2.1...')
gliner_model = GLiNER.from_pretrained('urchade/gliner_multi-v2.1', load_tokenizer=True, resize_token_embeddings=True)

end_cell('Cell 1: Setup & Ontology', {
    'Node types': len(ONTOLOGY['node_types']),
    'Edge types': len(ONTOLOGY['edge_types']),
    'NER labels': len(ONTOLOGY['ner_labels']),
    'Log file': LOG_FILE
})



<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


INFO     | ============================================================
INFO     | START: Cell 1: Setup & Ontology
INFO     | ============================================================
INFO     | Config: NEO4J=bolt://localhost:7687, WEAVIATE=localhost:8081
INFO     | LLM Model: nvidia/nemotron-3-super-120b-a12b:free
DEBUG    | OpenRouter API key loaded: YES
INFO     | Ontology loaded: 17 node types, 16 edge types
INFO     | Loading spaCy en_core_web_sm...
INFO     | Loading GLiNER urchade/gliner_multi-v2.1...


c:\Users\rizky_11yf1be\Desktop\Tugas_Akhir\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO     | ────────────────────────────────────────────────────────────
INFO     |   Node types: 17
INFO     |   Edge types: 16
INFO     |   NER labels: 9
INFO     |   Log file: logs\kg_pipeline_20260326_003313.log
INFO     | ✅ Cell 1: Setup & Ontology completed in 21.7s
INFO     | ============================================================



In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2: Backbone from Structured Data (Strwythura Part 1)
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 2: Backbone Construction')

df_papers = pd.read_csv('file_tabulars/dosen_papers_scholar_colab.csv').fillna('')
df_dosen = pd.read_csv('file_tabulars/dosen_infokom_final.csv').fillna('')
logger.info(f'CSV loaded: {len(df_papers)} papers, {len(df_dosen)} dosen')

MAX_PAPERS = 500
df_with_abstract = df_papers[df_papers['Abstract'].str.len() > 50]
df_sample = df_with_abstract.sample(n=min(MAX_PAPERS, len(df_with_abstract)), random_state=42).copy()
logger.info(f'Filtered: {len(df_with_abstract)} with abstracts, sampled {len(df_sample)}')

dosen_lookup = {
    str(r['scholar_id']).strip(): r.to_dict()
    for _, r in df_dosen.iterrows()
    if str(r['scholar_id']).strip() and str(r['scholar_id']).strip().lower() != 'nan'
}

nodes, edges = {}, []
paper_abstracts, paper_titles = {}, {}

# Dosen -> ProgramStudi -> Fakultas
dosen_count = 0
for _, row in df_dosen.iterrows():
    name = row.get('nama_norm', '') or row.get('nama_dosen', '')
    if not name: continue
    dosen_count += 1
    sid = str(row.get('scholar_id', '')).strip()
    d_id = f'dosen_{sid}' if sid and sid.lower() != 'nan' else f'dosen_{md5(name)}'
    prodi = str(row.get('prodi', '')).strip() or 'Unknown'
    nodes[d_id] = {'_label': 'Dosen', 'name': name, 'prodi': prodi,
                   'scholar_id': sid, 'nip': str(row.get('nip', '')), 'nidn': str(row.get('nidn', ''))}
    p_id = f'prodi_{normalize_text(prodi).replace(" ", "_")}'
    if p_id not in nodes: nodes[p_id] = {'_label': 'ProgramStudi', 'name': prodi}
    edges.append((d_id, p_id, 'MEMBER_OF', {}))
    fak = PRODI_FAKULTAS.get(prodi, 'Unknown')
    f_id = f'fak_{normalize_text(fak).replace(" ", "_")}'
    if f_id not in nodes: nodes[f_id] = {'_label': 'Fakultas', 'name': fak}
    edges.append((p_id, f_id, 'PART_OF', {}))

logger.info(f'Dosen backbone: {dosen_count} dosen registered')

# Papers -> Year, Journal, Authors, Keywords
paper_count, ext_author_count, keyword_count = 0, 0, 0
for _, row in df_sample.iterrows():
    t = str(row['Title']).strip()
    a = str(row['Abstract']).strip()
    y = str(row.get('Year', ''))[:4]
    journal = str(row.get('Journal', '')).strip()
    link = str(row.get('Link', '')); doi = str(row.get('DOI', '')).strip()
    if ('scholar' in link.lower() or not link or link == 'nan') and doi and doi != 'nan':
        link = f'https://doi.org/{doi}'
    pid = f'paper_{md5(t)}'
    nodes[pid] = {'_label': 'Paper', 'title': t, 'year': y, 'url': link, 'journal': journal}
    paper_abstracts[pid] = a; paper_titles[pid] = t
    paper_count += 1
    if y and y != 'nan':
        yid = f'year_{y}'
        if yid not in nodes: nodes[yid] = {'_label': 'Year', 'value': y}
        edges.append((pid, yid, 'PUBLISHED_YEAR', {}))
    j_clean = journal.split(',')[0].strip() if journal and journal != 'nan' else ''
    if j_clean:
        jid = f'journal_{md5(j_clean)}'
        if jid not in nodes: nodes[jid] = {'_label': 'Journal', 'name': j_clean}
        edges.append((pid, jid, 'PUBLISHED_IN', {}))
    authors = [x.strip() for x in str(row['Authors']).split(',') if x.strip()]
    aids = [x.strip() for x in str(row.get('Author IDs', '')).split(';') if x.strip()]
    for i, aname in enumerate(authors):
        asid = aids[i] if i < len(aids) else ''
        did = f'dosen_{asid}' if asid and asid in dosen_lookup else f'ext_{md5(aname)}'
        if did not in nodes:
            nodes[did] = {'_label': 'Dosen' if 'dosen_' in did else 'ExternalAuthor', 'name': aname}
            if 'ext_' in did: ext_author_count += 1
        edges.append((did, pid, 'WRITES', {'position': 'first' if i == 0 else 'co'}))
    kw_raw = str(row.get('Keywords', ''))
    if kw_raw and kw_raw != 'nan':
        for kw in re.split(r'[;,]', kw_raw):
            kw = kw.strip()
            if kw and len(kw) > 2:
                kwid = f'keyword_{md5(kw)}'
                if kwid not in nodes:
                    nodes[kwid] = {'_label': 'Keyword', 'name': kw}
                    keyword_count += 1
                edges.append((pid, kwid, 'HAS_KEYWORD', {}))

end_cell('Cell 2: Backbone Construction', {
    'Papers': paper_count, 'Dosen': dosen_count,
    'External Authors': ext_author_count, 'Keywords': keyword_count,
    'Total nodes': len(nodes), 'Total edges': len(edges)
})



INFO     | ============================================================
INFO     | START: Cell 2: Backbone Construction
INFO     | ============================================================
INFO     | CSV loaded: 5573 papers, 129 dosen
INFO     | Filtered: 311 with abstracts, sampled 311
INFO     | Dosen backbone: 129 dosen registered
INFO     | ────────────────────────────────────────────────────────────
INFO     |   Papers: 311
INFO     |   Dosen: 129
INFO     |   External Authors: 0
INFO     |   Keywords: 721
INFO     |   Total nodes: 1327
INFO     |   Total edges: 2691
INFO     | ✅ Cell 2: Backbone Construction completed in 0.6s
INFO     | ============================================================



In [3]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3: Text Parsing + NER (Strwythura Part 3)
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 3: Text Parsing + NER')

GLINER_SET1 = ['method', 'algorithm', 'evaluation metric', 'software', 'model']
GLINER_SET2 = ['dataset', 'problem', 'task', 'tool', 'technology', 'framework', 'field']
GLINER_THRESHOLD = 0.15

entity_store = OrderedDict()
entity_sequences = []
entity_uid = 0

SRC_NER = 0    # GLiNER NER (highest priority)
SRC_TITLE = 1  # Title regex
SRC_CSV = 2    # CSV keywords

extracted_entities = {}
chunk_vdb = []
_ner_count, _title_count, _csv_count = 0, 0, 0

def make_lemma_key(text):
    # Strwythura-style POS.lemma key (from nlp.py tokenize_lemma)
    doc = nlp(str(text).strip())
    parts = []
    for tok in doc:
        pos = 'NOUN' if tok.pos_ in ('PROPN', 'NOUN') else tok.pos_
        if pos in ('NOUN', 'ADJ', 'VERB'):
            parts.append(f'{pos}.{tok.lemma_.lower()}')
    return ' '.join(parts) if parts else normalize_text(text)

def register_entity(text, label, source_priority):
    # Register entity in store with Strwythura-style dedup by lemma_key.
    global entity_uid, _ner_count, _title_count, _csv_count
    lemma_key = make_lemma_key(text)
    if not lemma_key or len(lemma_key) < 3:
        return None
    mapped_label = ONTOLOGY['ner_label_map'].get(label.lower(), 'Field')
    if lemma_key not in entity_store:
        entity_store[lemma_key] = {
            'uid': entity_uid, 'text': text.strip(), 'label': mapped_label,
            'count': 1, 'source': source_priority, 'description': ''
        }
        entity_uid += 1
        logger.debug(f'NEW entity [{mapped_label}]: "{text.strip()}" -> lemma_key="{lemma_key}" (src={source_priority})')
        if source_priority == SRC_NER: _ner_count += 1
        elif source_priority == SRC_TITLE: _title_count += 1
        else: _csv_count += 1
    elif source_priority < entity_store[lemma_key]['source']:
        old_src = entity_store[lemma_key]['source']
        entity_store[lemma_key]['text'] = text.strip()
        entity_store[lemma_key]['label'] = mapped_label
        entity_store[lemma_key]['source'] = source_priority
        entity_store[lemma_key]['count'] += 1
        logger.debug(f'PROMOTED entity: "{text.strip()}" source {old_src} -> {source_priority}')
    else:
        entity_store[lemma_key]['count'] += 1
    return lemma_key

total = len(paper_abstracts)
gliner_errors = 0
for i, (pid, abstract) in enumerate(paper_abstracts.items()):
    title = paper_titles.get(pid, '')
    full_text = f'{title}. {abstract}'
    paper_node = nodes.get(pid, {})
    chunk_vdb.append({'title': title, 'content': full_text, 'year': paper_node.get('year',''), 'paperUrl': paper_node.get('url','')})
    paper_lemma_keys = []
    # Pass 1: GLiNER NER
    input_text = full_text[:2000]
    for label_set in [GLINER_SET1, GLINER_SET2]:
        try:
            ents = gliner_model.predict_entities(input_text, label_set, threshold=GLINER_THRESHOLD)
            for e in ents:
                lk = register_entity(e['text'], e['label'], SRC_NER)
                if lk: paper_lemma_keys.append(lk)
        except Exception as ex:
            gliner_errors += 1
            logger.debug(f'GLiNER error on paper {pid}: {type(ex).__name__}: {ex}')
    # Pass 2: Title regex (abbreviations + multi-word)
    for term in re.findall(r'[A-Z]{2,}[0-9]*', title):
        lk = register_entity(term, 'method', SRC_TITLE)
        if lk: paper_lemma_keys.append(lk)
    for term in re.findall(r'[A-Z][A-Za-z0-9]*(?:\s+[A-Z][A-Za-z0-9]*)+', title):
        lk = register_entity(term, 'method', SRC_TITLE)
        if lk: paper_lemma_keys.append(lk)
    # Pass 3: CSV Keywords
    paper_row = df_sample[df_sample['Title'].str.strip() == title]
    if not paper_row.empty:
        kw_raw = str(paper_row.iloc[0].get('Keywords', ''))
        if kw_raw and kw_raw != 'nan':
            for kw in re.split(r'[;,]', kw_raw):
                kw = kw.strip()
                if kw and len(kw) > 2:
                    lk = register_entity(kw, 'field', SRC_CSV)
                    if lk: paper_lemma_keys.append(lk)
    extracted_entities[pid] = list(set(paper_lemma_keys))
    entity_sequences.append([entity_store[lk]['uid'] for lk in paper_lemma_keys if lk in entity_store])
    if (i + 1) % 50 == 0:
        logger.info(f'  Parsed {i+1}/{total} papers | unique entities so far: {len(entity_store)}')

label_dist = defaultdict(int)
for e in entity_store.values(): label_dist[e['label']] += 1

end_cell('Cell 3: Text Parsing + NER', {
    'Unique entities (lemma-key deduped)': len(entity_store),
    'From NER': _ner_count, 'From Title Regex': _title_count, 'From CSV': _csv_count,
    'GLiNER errors': gliner_errors,
    'Label distribution': dict(label_dist)
})



INFO     | ============================================================
INFO     | START: Cell 3: Text Parsing + NER
INFO     | ============================================================
DEBUG    | NEW entity [Tool]: "Automated Assessment Tool" -> lemma_key="NOUN.automated NOUN.assessment NOUN.tool" (src=0)
DEBUG    | NEW entity [Tool]: "automated assessment tool" -> lemma_key="VERB.automate NOUN.assessment NOUN.tool" (src=0)
DEBUG    | NEW entity [Model]: "software developments models" -> lemma_key="NOUN.software NOUN.development NOUN.model" (src=0)
DEBUG    | NEW entity [Model]: "waterfall model" -> lemma_key="NOUN.waterfall NOUN.model" (src=0)
DEBUG    | NEW entity [Model]: "correlation" -> lemma_key="NOUN.correlation" (src=0)
DEBUG    | NEW entity [Field]: "Computer Programming" -> lemma_key="NOUN.computer NOUN.programming" (src=0)
DEBUG    | NEW entity [Field]: "digital storytelling" -> lemma_key="ADJ.digital NOUN.storytelling" (src=0)
DEBUG    | NEW entity [Tool]: "software dev

In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4: Entity Resolution (Strwythura Part 1 + 4)
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 4: Entity Resolution (3-Layer)')

alias_map = {}

# ── Layer 2: Abbreviation Extraction from abstracts ──
abbr_pat1 = re.compile(r'([A-Za-z][\w\s\-]+?)\s*\(([A-Z][A-Za-z0-9\s\-\.]*?)\)')
abbr_pat2 = re.compile(r'([A-Z][A-Z0-9]{1,10})\s*\(([A-Za-z][\w\s\-]+?)\)')

abbr_found = 0
for pid, abstract in paper_abstracts.items():
    full = f'{paper_titles.get(pid, "")}. {abstract}'
    for pat in [abbr_pat1, abbr_pat2]:
        for m in pat.finditer(full):
            a, b = m.group(1).strip(), m.group(2).strip()
            short, long_ = (a, b) if len(a) < len(b) else (b, a)
            if len(short) >= 2:
                lk_short = make_lemma_key(short)
                lk_long = make_lemma_key(long_)
                if lk_short and lk_long and lk_short != lk_long:
                    alias_map[lk_short] = lk_long
                    abbr_found += 1
                    logger.debug(f'ABBREV alias: "{short}" -> "{long_}"')

logger.info(f'Layer 2 (Regex): {len(alias_map)} abbreviation aliases from {abbr_found} matches')

# ── Layer 3: LLM Synonym Clustering ──
unique_texts = [e['text'] for e in entity_store.values()]
BATCH = 80
CLUSTER_PROMPT = (
    "Kamu ahli Entity Resolution untuk KG akademik.\n"
    "Kelompokkan entitas yang BERMAKNA SAMA (sinonim, singkatan, terjemahan).\n\n"
    'Contoh: "QoS" = "Quality of Service" | "CNN" = "Convolutional Neural Network" | "akurasi" = "accuracy"\n\n'
    "Daftar entitas:\n{ents}\n\n"
    'Output JSON: {{"groups": [{{"canonical": "English standard name", "members": ["var1", "var2"]}}]}}\n'
    "Hanya kelompokkan yang BENAR-BENAR sinonim. Entitas unik tidak perlu dimasukkan."
)

llm_clusters = 0
llm_batch_count = 0
for start in range(0, len(unique_texts), BATCH):
    batch = unique_texts[start:start + BATCH]
    if len(batch) < 2: continue
    llm_batch_count += 1
    ents_str = '\n'.join([f'- {e}' for e in batch])
    result = call_openrouter(CLUSTER_PROMPT.format(ents=ents_str))
    time.sleep(0.3)
    for grp in result.get('groups', []):
        if not isinstance(grp, dict): continue
        canonical = grp.get('canonical', '')
        members = grp.get('members', [])
        if canonical and len(members) > 1:
            lk_canon = make_lemma_key(canonical)
            for member in members:
                lk_mem = make_lemma_key(member)
                if lk_mem and lk_mem != lk_canon:
                    alias_map[lk_mem] = lk_canon
                    logger.debug(f'LLM synonym: "{member}" -> "{canonical}"')
            llm_clusters += 1
    if llm_batch_count % 5 == 0:
        logger.info(f'  LLM clustering batch {llm_batch_count}/{(len(unique_texts)//BATCH)+1}...')

logger.info(f'Layer 3 (LLM): {llm_clusters} synonym clusters from {llm_batch_count} batches')

# ── Apply alias resolution ──
def resolve(lk):
    visited = set()
    while lk in alias_map and lk not in visited:
        visited.add(lk)
        lk = alias_map[lk]
    return lk

merged_count = 0
for pid in extracted_entities:
    resolved = []
    for lk in extracted_entities[pid]:
        canon = resolve(lk)
        if canon != lk:
            merged_count += 1
            logger.debug(f'MERGED: "{lk}" -> "{canon}" in paper {pid}')
        resolved.append(canon)
    extracted_entities[pid] = list(set(resolved))

end_cell('Cell 4: Entity Resolution (3-Layer)', {
    'Total alias mappings': len(alias_map),
    'Entity references merged': merged_count,
    'Abbreviation aliases (Layer 2)': abbr_found,
    'LLM synonym clusters (Layer 3)': llm_clusters
})



INFO     | ============================================================
INFO     | START: Cell 4: Entity Resolution (3-Layer)
INFO     | ============================================================
DEBUG    | ABBREV alias: "Equal-Cost Multipath" -> "Penerapan Load Balancing Dengan Metode Ecmp"
DEBUG    | ABBREV alias: "Quality of Services" -> "Perlunya menganalisis bagaimana peningkatan QoS"
DEBUG    | ABBREV alias: "Equal-Cost Multipath" -> "melalui penerapan metode Load Balancing Metode ECMP"
DEBUG    | ABBREV alias: "ECMP" -> "Equal-Cost Multipath"
DEBUG    | ABBREV alias: "Ela" -> "Klasifikasi Gambar Asli Dan Manipulasi Menggunakan Error Level Analysis"
DEBUG    | ABBREV alias: "Cnn" -> "Sebagai Proses Komputasi Metode Convolutional Neural Network"
DEBUG    | ABBREV alias: "ELA" -> "salah satunya dengan menentukan kualitas hasil tingkat kompresi gambar pada mekanisme error level analysis"
DEBUG    | ABBREV alias: "CNN" -> "convolutional neural network"
DEBUG    | ABBREV alias: "MLP

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5: LLM Curation (Strwythura Part 4 — HITL replacement)
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 5: LLM Curation')

entity_vdb, relationship_vdb, keywords_vdb = [], [], []
entity_node_map = {}
VALID_LABELS = set(ONTOLOGY['node_types'].keys()) - {'Dosen','ExternalAuthor','Paper','ProgramStudi','Fakultas','Journal','Year','Keyword'}
logger.info(f'Valid entity labels for curation: {VALID_LABELS}')

CURATION_PROMPT = (
    "Kamu ahli NLP. Validasi dan perkaya entitas dari paper akademik ini.\n\n"
    "## Judul: {title}\n"
    "## Abstrak: {abstract}\n"
    "## Entitas terdeteksi NER (mungkin noisy): {entities}\n\n"
    "## Tugas:\n"
    "1. VALIDASI: hapus yang bukan entitas ilmiah (kata umum).\n"
    "2. PERBAIKI label jika salah. Label valid: Method, Model, Metric, Dataset, Problem, Task, Field, Tool, Innovation\n"
    "3. TAMBAHKAN entitas penting yang terlewat oleh NER.\n"
    "4. Berikan DESCRIPTION 1 kalimat per entitas (untuk vector search).\n"
    "5. Ekstrak RELASI antar entitas: USES (source uses target), ADDRESSES, PROPOSES\n\n"
    "## Output JSON:\n"
    '{{"entities": [{{"text": "exact text", "label": "Method", "description": "1 sentence"}}],\n'
    '  "relations": [{{"source": "entity1", "target": "entity2", "relation": "USES", "description": "1 sentence"}}]}}\n\n'
    "PENTING: Entitas harus text spans yang ADA di abstrak/judul. Minimal 3 entitas per abstrak."
)

total = len(extracted_entities)
llm_calls, llm_errors, skipped = 0, 0, 0
curated_ent_count, curated_rel_count = 0, 0

for i, (pid, lemma_keys) in enumerate(extracted_entities.items()):
    abstract = paper_abstracts.get(pid, '')
    title = paper_titles.get(pid, '')
    if len(abstract) < 50:
        skipped += 1
        logger.debug(f'SKIPPED paper {pid}: abstract too short ({len(abstract)} chars)')
        continue
    ent_hints = [{'text': entity_store[lk]['text'], 'label': entity_store[lk]['label']}
                 for lk in lemma_keys if lk in entity_store]
    enriched = call_openrouter(CURATION_PROMPT.format(
        title=title, abstract=abstract[:2000],
        entities=json.dumps(ent_hints, ensure_ascii=False)
    ))
    llm_calls += 1; time.sleep(0.3)
    if not enriched:
        llm_errors += 1
        logger.warning(f'LLM returned empty for paper: "{title[:60]}..."')
        continue
    for ent in enriched.get('entities', []):
        if not isinstance(ent, dict): continue
        txt = str(ent.get('text', '')).strip()
        lbl = str(ent.get('label', 'Field')).strip()
        desc = str(ent.get('description', '')).strip()
        lbl_cap = lbl.capitalize()
        if lbl_cap not in VALID_LABELS:
            lbl_cap = ONTOLOGY['ner_label_map'].get(lbl.lower(), 'Field')
        if not txt or len(txt) < 2: continue
        lk = resolve(make_lemma_key(txt))
        nid = f'{lbl_cap.lower()}_{md5(lk)}'
        if lk not in entity_node_map:
            entity_node_map[lk] = nid
            nodes[nid] = {'_label': lbl_cap, 'name': txt, 'description': desc}
            entity_vdb.append({'nodeId': nid, 'entityName': txt, 'entityType': lbl_cap, 'description': desc})
            curated_ent_count += 1
            logger.debug(f'CURATED entity [{lbl_cap}]: "{txt}" -> {nid}')
            if lk in entity_store:
                entity_store[lk]['description'] = desc
        edges.append((pid, entity_node_map[lk], f'HAS_{lbl_cap.upper()}', {}))
    for rel in enriched.get('relations', []):
        if not isinstance(rel, dict): continue
        slk = resolve(make_lemma_key(rel.get('source','')))
        tlk = resolve(make_lemma_key(rel.get('target','')))
        if slk in entity_node_map and tlk in entity_node_map:
            rtype = str(rel.get('relation','USES')).upper().replace(' ','_')
            rdesc = str(rel.get('description',''))
            edges.append((entity_node_map[slk], entity_node_map[tlk], rtype, {'description': rdesc}))
            relationship_vdb.append({'srcId': entity_node_map[slk], 'tgtId': entity_node_map[tlk], 'relType': rtype, 'description': rdesc})
            curated_rel_count += 1
            logger.debug(f'RELATION: {entity_node_map[slk]} -[{rtype}]-> {entity_node_map[tlk]}')
    kws = [entity_store[lk]['text'] for lk in lemma_keys if lk in entity_store]
    if kws: keywords_vdb.append({'keywords': '; '.join(kws), 'sourcePaper': pid})
    if (i+1) % 50 == 0:
        logger.info(f'  Curated {i+1}/{total} | entities: {curated_ent_count} | relations: {curated_rel_count} | errors: {llm_errors}')

end_cell('Cell 5: LLM Curation', {
    'Curated entities': curated_ent_count,
    'Curated relations': curated_rel_count,
    'LLM calls': llm_calls, 'LLM errors': llm_errors,
    'Skipped (short abstract)': skipped,
    'Total nodes now': len(nodes), 'Total edges now': len(edges)
})



In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6: Distillation & Entity Embeddings (Strwythura Part 5)
# ══════════════════════════════════════════════════════════════════════
import gensim
start_cell('Cell 6: Distillation & Embeddings')

high_centrality = {lk for lk, e in entity_store.items() if e['count'] >= 2}
low_centrality = {lk for lk, e in entity_store.items() if e['count'] < 2}
logger.info(f'Centrality filter: {len(high_centrality)} high (count>=2), {len(low_centrality)} low (count<2)')

w2v_sentences = []
for seq in entity_sequences:
    if len(seq) >= 2:
        w2v_sentences.append([str(uid) for uid in seq])

logger.info(f'Word2Vec training sentences: {len(w2v_sentences)}')

if w2v_sentences:
    w2v_model = gensim.models.Word2Vec(
        sentences=w2v_sentences,
        vector_size=32,
        window=max(len(s) for s in w2v_sentences),
        min_count=1,
        sg=1,
    )
    w2v_model.save('file_tabulars/entity_embeddings.w2v')
    logger.info(f'Entity embeddings trained: {len(w2v_model.wv)} vectors, dim={w2v_model.wv.vector_size}')
    uid_to_lk = {str(e['uid']): lk for lk, e in entity_store.items()}
    for test_uid in list(w2v_model.wv.index_to_key)[:3]:
        similar = w2v_model.wv.most_similar(test_uid, topn=3)
        test_name = entity_store.get(uid_to_lk.get(test_uid, ''), {}).get('text', '?')
        sim_names = [(entity_store.get(uid_to_lk.get(s[0], ''), {}).get('text', '?'), round(s[1], 3)) for s in similar]
        logger.info(f'  Similar to "{test_name}": {sim_names}')
else:
    logger.warning('Not enough entity sequences for Word2Vec training')

end_cell('Cell 6: Distillation & Embeddings', {
    'High centrality entities': len(high_centrality),
    'Low centrality (filtered)': len(low_centrality),
    'W2V sentences': len(w2v_sentences),
    'Total nodes': len(nodes), 'Total edges': len(edges)
})



In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7: Database Ingestion (Neo4j + Weaviate)
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 7: Database Ingestion')

# Weaviate
logger.info(f'Connecting to Weaviate at {WEAVIATE_HOST}:{WEAVIATE_PORT}...')
wv_client = weaviate.connect_to_local(host=WEAVIATE_HOST, port=WEAVIATE_PORT)
logger.info('Weaviate connected.')

classes_config = {
    'EntityEmbedding': [Property(name='entityName', data_type=DataType.TEXT), Property(name='entityType', data_type=DataType.TEXT), Property(name='description', data_type=DataType.TEXT), Property(name='nodeId', data_type=DataType.TEXT)],
    'RelationshipEmbedding': [Property(name='srcId', data_type=DataType.TEXT), Property(name='tgtId', data_type=DataType.TEXT), Property(name='relType', data_type=DataType.TEXT), Property(name='description', data_type=DataType.TEXT)],
    'ContentKeyword': [Property(name='keywords', data_type=DataType.TEXT), Property(name='sourcePaper', data_type=DataType.TEXT)],
    'PaperChunk': [Property(name='title', data_type=DataType.TEXT), Property(name='content', data_type=DataType.TEXT), Property(name='year', data_type=DataType.TEXT), Property(name='paperUrl', data_type=DataType.TEXT)],
}
for c, props in classes_config.items():
    if wv_client.collections.exists(c):
        wv_client.collections.delete(c)
        logger.info(f'  Deleted existing Weaviate collection: {c}')
    wv_client.collections.create(name=c, vectorizer_config=Configure.Vectorizer.text2vec_transformers(), properties=props)
    logger.info(f'  Created Weaviate collection: {c}')

# Neo4j
logger.info(f'Connecting to Neo4j at {NEO4J_URI}...')
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
ALL_LABELS = list(ONTOLOGY['node_types'].keys())
with driver.session() as s:
    s.run('MATCH (n) DETACH DELETE n')
    logger.info('  Neo4j: all existing data deleted')
    for lbl in ALL_LABELS:
        s.run(f'CREATE CONSTRAINT IF NOT EXISTS FOR (n:{lbl}) REQUIRE n.node_id IS UNIQUE')
    logger.info(f'  Neo4j: {len(ALL_LABELS)} uniqueness constraints created')

# Neo4j: nodes
logger.info(f'Ingesting {len(nodes)} nodes to Neo4j...')
node_errors = 0
t0 = time.time()
with driver.session() as s:
    for nid, d in nodes.items():
        try:
            lbl = d['_label']
            props = {k: str(v) for k, v in d.items() if k != '_label' and v and str(v) != 'nan'}
            props['node_id'] = nid
            sc = ', '.join([f'n.`{k}` = ${k}' for k in props])
            s.run(f'MERGE (n:{lbl} {{node_id: $node_id}}) SET {sc}', **props)
        except Exception as e:
            node_errors += 1
            logger.error(f'Node insert error [{nid}]: {type(e).__name__}: {e}')
logger.info(f'  Neo4j nodes ingested in {time.time()-t0:.1f}s (errors: {node_errors})')

# Neo4j: edges
logger.info(f'Ingesting {len(edges)} edges to Neo4j...')
edge_errors, edge_skipped = 0, 0
t0 = time.time()
with driver.session() as s:
    for src, tgt, rel, props in edges:
        if src not in nodes or tgt not in nodes:
            edge_skipped += 1
            continue
        try:
            pc = ''
            if props:
                sp = [f'r.`{k}` = "{v}"' for k, v in props.items() if v]
                if sp: pc = ' SET ' + ', '.join(sp)
            s.run(f'MATCH (a {{node_id: $s}}) MATCH (b {{node_id: $t}}) MERGE (a)-[r:{rel}]->(b){pc}', s=src, t=tgt)
        except Exception as e:
            edge_errors += 1
            logger.error(f'Edge insert error [{src}]-[{rel}]->[{tgt}]: {type(e).__name__}: {e}')
logger.info(f'  Neo4j edges ingested in {time.time()-t0:.1f}s (errors: {edge_errors}, skipped: {edge_skipped})')

# Weaviate: batched insert
logger.info('Ingesting to Weaviate (batched)...')
BATCH_SIZE = 50
def batch_insert(name, data):
    col = wv_client.collections.get(name)
    batch_errors = 0
    for start in range(0, len(data), BATCH_SIZE):
        try:
            col.data.insert_many([DataObject(properties=item) for item in data[start:start+BATCH_SIZE]])
        except Exception as e:
            batch_errors += 1
            logger.error(f'Weaviate batch error [{name}] at offset {start}: {type(e).__name__}: {e}')
    logger.info(f'  {name}: {len(data)} objects (batch errors: {batch_errors})')
if entity_vdb: batch_insert('EntityEmbedding', entity_vdb)
if relationship_vdb: batch_insert('RelationshipEmbedding', relationship_vdb)
if keywords_vdb: batch_insert('ContentKeyword', keywords_vdb)
if chunk_vdb: batch_insert('PaperChunk', chunk_vdb)

# Stats
with driver.session() as s:
    nc = s.run('MATCH (n) RETURN count(n) as c').single()['c']
    ec = s.run('MATCH ()-[r]->() RETURN count(r) as c').single()['c']
    lc = s.run('MATCH (n) RETURN labels(n)[0] as label, count(n) as cnt ORDER BY cnt DESC').data()
    rc = s.run('MATCH ()-[r]->() RETURN type(r) as relType, count(r) as cnt ORDER BY cnt DESC').data()

sep = '=' * 60
logger.info(f'\n{sep}')
logger.info(f'KG CONSTRUCTION PIPELINE COMPLETE!')
logger.info(f'{sep}')
logger.info(f'Neo4j: {nc} nodes, {ec} edges')
logger.info(f'--- Node Distribution ---')
for row in lc: logger.info(f'  {row["label"]:20s}: {row["cnt"]}')
logger.info(f'--- Edge Distribution ---')
for row in rc: logger.info(f'  {row["relType"]:20s}: {row["cnt"]}')

wv_client.close(); driver.close()
end_cell('Cell 7: Database Ingestion', {
    'Neo4j nodes': nc, 'Neo4j edges': ec,
    'Node insert errors': node_errors, 'Edge insert errors': edge_errors,
    'Edge skipped (missing src/tgt)': edge_skipped
})



In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8: Verification Queries
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 8: Verification Queries')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

queries = [
    ('1. Metode Terpopuler', 'MATCH (p:Paper)-[:HAS_METHOD]->(m:Method) RETURN m.name AS Metode, COUNT(p) AS Jumlah ORDER BY Jumlah DESC LIMIT 10'),
    ('2. Paper Mirip (Metode Sama)', 'MATCH (p1:Paper)-[:HAS_METHOD]->(m:Method)<-[:HAS_METHOD]-(p2:Paper) WHERE p1 <> p2 RETURN p1.title AS Paper1, m.name AS Metode, p2.title AS Paper2 LIMIT 10'),
    ('3. Topik per Prodi', 'MATCH (ps:ProgramStudi)<-[:MEMBER_OF]-(d:Dosen)-[:WRITES]->(p:Paper)-[:IN_FIELD]->(f:Field) RETURN ps.name AS Prodi, f.name AS Field, COUNT(p) AS Total ORDER BY Total DESC LIMIT 10'),
    ('4. Kolaborasi Lintas Prodi', 'MATCH (d1:Dosen)-[:WRITES]->(p:Paper)<-[:WRITES]-(d2:Dosen) MATCH (d1)-[:MEMBER_OF]->(ps1) MATCH (d2)-[:MEMBER_OF]->(ps2) WHERE elementId(d1)<elementId(d2) AND ps1<>ps2 RETURN d1.name AS D1, ps1.name AS P1, d2.name AS D2, ps2.name AS P2, COUNT(p) AS Papers ORDER BY Papers DESC LIMIT 5'),
    ('5. Tren Model per Tahun', 'MATCH (p:Paper)-[:HAS_MODEL]->(m:Model) MATCH (p)-[:PUBLISHED_YEAR]->(y:Year) RETURN y.value AS Tahun, m.name AS Model, COUNT(p) AS Jumlah ORDER BY Tahun DESC, Jumlah DESC LIMIT 10'),
    ('6. Dosen Produktif & Spesialisasi', 'MATCH (d:Dosen)-[:WRITES]->(p:Paper)-[:IN_FIELD]->(f:Field) RETURN d.name AS Dosen, f.name AS Topik, COUNT(p) AS Total ORDER BY Total DESC LIMIT 10'),
    ('7. Problem vs Tool', 'MATCH (t:Tool)<-[:HAS_TOOL]-(p:Paper)-[:ADDRESSES]->(pr:Problem) RETURN pr.name AS Masalah, t.name AS Tool, COUNT(p) AS Kasus ORDER BY Kasus DESC LIMIT 10'),
    ('8. Metrik per Field', 'MATCH (f:Field)<-[:IN_FIELD]-(p:Paper)-[:HAS_METRIC]->(m:Metric) RETURN f.name AS Field, m.name AS Metrik, COUNT(p) AS Freq ORDER BY Freq DESC LIMIT 10'),
    ('9. Paper Kompleks', 'MATCH (p:Paper)-[:HAS_METHOD|HAS_MODEL]->(e) WITH p, COUNT(e) AS K, COLLECT(e.name) AS List WHERE K>1 RETURN p.title AS Paper, K, List ORDER BY K DESC LIMIT 5'),
    ('10. Pencarian Pakar Sebidang', 'MATCH (d1:Dosen)-[:WRITES]->(p1:Paper)-[:ADDRESSES]->(pr:Problem)<-[:ADDRESSES]-(p2:Paper)<-[:WRITES]-(d2:Dosen) WHERE d1<>d2 AND elementId(d1)<elementId(d2) RETURN pr.name AS Problem, d1.name AS Peneliti1, d2.name AS Peneliti2 LIMIT 10'),
]

passed, failed = 0, 0
for title, q in queries:
    logger.info(f'\n--- {title} ---')
    logger.debug(f'Query: {q}')
    with driver.session() as s:
        try:
            result = s.run(q).data()
            if result:
                passed += 1
                logger.info(f'  ✅ {len(result)} rows returned')
                display(pd.DataFrame(result))
            else:
                failed += 1
                logger.warning(f'  ⚠️ No results')
        except Exception as e:
            failed += 1
            logger.error(f'  ❌ Query error: {type(e).__name__}: {e}')

driver.close()

end_cell('Cell 8: Verification Queries', {
    'Queries passed': passed, 'Queries with no results': failed,
    'Log file': LOG_FILE
})

logger.info(f'Full debug log saved to: {LOG_FILE}')

